# WaveFollower Training Notebook
## Trend-Following Multi-Timeframe Model — GPU Training

**WaveFollower** is a pair-agnostic trend-following model that:
- Classifies trend direction (UP/DOWN/NEUTRAL) across 4 timeframes
- Enters on pullbacks (HL in uptrend, LH in downtrend)
- Doubles up on valid pullback continuations (`add_score`)
- Exits on structure breaks (LL in uptrend, HH in downtrend)

**Data**: Reuses the same processed parquet data from the MTF pipeline — identical features (ohlcv, structure, rsi, volume). No reprocessing needed.

**OANDA Account**: `101-001-30902818-002` (separate from MTF's `-001`)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Imports & GPU Check
# ═══════════════════════════════════════════════════════════════
import sys, os, time, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler

# Ensure wavetrader package is importable
sys.path.insert(0, str(Path.cwd()))

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
else:
    print("WARNING: No GPU detected — training will use CPU (slow).")

## 2. Configuration & Data Paths

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════
from wavetrader.wave_follower import WaveFollower, WaveFollowerConfig

config = WaveFollowerConfig(
    # --- Timeframes (same 4 as MTF) ---
    # timeframes=["15min", "1h", "4h", "1d"],
    # lookbacks={"15min": 120, "1h": 120, "4h": 100, "1d": 60},

    # --- Training hyperparams ---
    learning_rate=5e-5,
    batch_size=16,       # Increase to 32/64 if VRAM allows
    epochs=60,
    dropout=0.15,
)

# Data paths — reuse MTF processed data (same features!)
PROCESSED_DIR = Path("processed_data")
RAW_DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints/wavefollower")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Pairs to train on (WaveFollower is pair-agnostic — train on ALL pairs)
PAIRS = ["GBPJPY", "EURJPY", "USDJPY", "GBPUSD"]
PAIR_NAMES = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY", "USDJPY": "USD/JPY", "GBPUSD": "GBP/USD"}

print(f"Config: {config.timeframes}, lookbacks={config.lookbacks}")
print(f"Batch size: {config.batch_size}, LR: {config.learning_rate}, Epochs: {config.epochs}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## 3. Check & Load Processed Data

The MTF model already processed all 4 pairs × 4 timeframes into `processed_data/{train,val,test}/`. WaveFollower uses the **exact same features** (ohlcv_norm, structure_0-7, rsi_norm, volume_norm), so we load directly from there — no reprocessing needed.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Check what processed data exists
# ═══════════════════════════════════════════════════════════════
from wavetrader.data import load_forex_data

def check_processed_data(processed_dir, pairs, timeframes):
    """Check availability of processed parquet files."""
    status = {}
    for split in ["train", "val", "test"]:
        split_dir = processed_dir / split
        for pair in pairs:
            for tf in timeframes:
                tf_short = tf.replace("min", "m")
                path = split_dir / f"{pair}_{tf_short}.parquet"
                key = f"{split}/{pair}_{tf_short}"
                status[key] = path.exists()
    
    found = sum(v for v in status.values())
    total = len(status)
    print(f"Processed data: {found}/{total} files found")
    
    # Check per-split summary
    for split in ["train", "val", "test"]:
        split_files = {k: v for k, v in status.items() if k.startswith(split)}
        n = sum(split_files.values())
        print(f"  {split}: {n}/{len(split_files)} files")
    
    missing = [k for k, v in status.items() if not v]
    if missing:
        print(f"\nMissing files:")
        for m in missing[:10]:
            print(f"  - {m}")
    
    return all(status.values()), status

all_found, data_status = check_processed_data(PROCESSED_DIR, PAIRS, config.timeframes)

if all_found:
    print("\n All processed data available — loading from processed_data/")
else:
    print("\n Some data missing — will load from raw CSV and process")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Load data from processed parquets (or raw CSV fallback)
# ═══════════════════════════════════════════════════════════════
from wavetrader.data import _normalise_df

def load_split_data(split, pairs, timeframes, processed_dir, raw_dir):
    """Load multi-TF DataFrames for all pairs in a split."""
    all_pair_data = {}
    split_dir = processed_dir / split
    
    for pair_tag in pairs:
        pair_name = PAIR_NAMES[pair_tag]
        dfs = {}
        for tf in timeframes:
            tf_short = tf.replace("min", "m")
            parquet_path = split_dir / f"{pair_tag}_{tf_short}.parquet"
            
            if parquet_path.exists():
                df = pd.read_parquet(parquet_path)
                df = _normalise_df(df)
                dfs[tf] = df
            else:
                # Fallback: load from raw CSV
                print(f"  Loading raw: {pair_name} {tf}")
                df = load_forex_data(pair_name, tf, data_dir=str(raw_dir))
                if df is not None and not df.empty:
                    dfs[tf] = df
                else:
                    print(f"  WARNING: No data for {pair_name} {tf}!")
        
        if len(dfs) == len(timeframes):
            all_pair_data[pair_tag] = dfs
            entry_len = len(dfs[timeframes[0]])
            print(f"  {pair_tag}: {entry_len:,} entry-TF bars, all {len(timeframes)} TFs loaded")
        else:
            print(f"  {pair_tag}: SKIPPED — only {len(dfs)}/{len(timeframes)} TFs available")
    
    return all_pair_data

# Load train/val/test splits
print("Loading TRAIN split:")
train_data = load_split_data("train", PAIRS, config.timeframes, PROCESSED_DIR, RAW_DATA_DIR)

print("\nLoading VAL split:")
val_data = load_split_data("val", PAIRS, config.timeframes, PROCESSED_DIR, RAW_DATA_DIR)

print("\nLoading TEST split:")
test_data = load_split_data("test", PAIRS, config.timeframes, PROCESSED_DIR, RAW_DATA_DIR)

print(f"\nTrain pairs: {list(train_data.keys())}")
print(f"Val pairs  : {list(val_data.keys())}")
print(f"Test pairs : {list(test_data.keys())}")

## 4. Data Format Comparison & Verification

Quick sanity check: the features produced by `prepare_features()` are identical for both models.
WaveFollower just uses slightly larger lookbacks (120 vs 100 on 15m/1h).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Verify data format compatibility
# ═══════════════════════════════════════════════════════════════
from wavetrader.dataset import prepare_features
from wavetrader.config import MTFConfig

# Compare what features each model expects
REQUIRED_FEATURES = {
    "ohlcv":     ["open_norm", "high_norm", "low_norm", "close_norm", "volume_norm"],
    "structure": [f"structure_{i}" for i in range(8)],
    "rsi":       ["rsi_norm", "rsi_delta_norm", "rsi_accel_norm"],
    "volume":    ["volume_norm", "volume_ratio", "volume_delta"],
}

# Take one sample pair and prepare features
sample_pair = list(train_data.keys())[0]
sample_tf = config.timeframes[0]
sample_df = train_data[sample_pair][sample_tf].copy()
prepared = prepare_features(sample_df, lookahead=10, pair=PAIR_NAMES[sample_pair])

print(f"Sample: {sample_pair} {sample_tf} — {len(prepared):,} rows after prepare_features()")
print(f"Columns ({len(prepared.columns)}): {list(prepared.columns)}")
print()

# Check all required features exist
all_ok = True
for group, cols in REQUIRED_FEATURES.items():
    present = all(c in prepared.columns for c in cols)
    status = "OK" if present else "MISSING"
    if not present:
        all_ok = False
        missing = [c for c in cols if c not in prepared.columns]
        print(f"  {group:12s}: {status} — missing: {missing}")
    else:
        print(f"  {group:12s}: {status} ({len(cols)} features)")

print(f"\nLookback comparison:")
mtf_cfg = MTFConfig()
for tf in config.timeframes:
    wf_lb = config.lookbacks[tf]
    mtf_lb = mtf_cfg.lookbacks[tf]
    print(f"  {tf:6s}: WaveFollower={wf_lb}, MTF={mtf_lb} {'(same)' if wf_lb == mtf_lb else ''}")

if all_ok:
    print("\n PASS: All features available — shared data is fully compatible")

## 5. Build Datasets & DataLoaders

WaveFollower is **pair-agnostic** — we train on ALL pairs combined, so the model learns pure structure patterns instead of pair-specific price levels. We concatenate all pairs into one large dataset.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Build MTFForexDataset for each pair, then combine
# ═══════════════════════════════════════════════════════════════
from wavetrader.dataset import MTFForexDataset, mtf_collate_fn
from torch.utils.data import ConcatDataset

def build_combined_dataset(pair_data, config, pairs, lookahead=10):
    """Build one dataset per pair, then concatenate them all."""
    datasets = []
    for pair_tag in pairs:
        if pair_tag not in pair_data:
            continue
        pair_name = PAIR_NAMES[pair_tag]
        ds = MTFForexDataset(
            dataframes=pair_data[pair_tag],
            config=config,
            lookahead=lookahead,
            threshold_pips=20.0,
            pair=pair_name,
        )
        datasets.append(ds)
        print(f"  {pair_tag}: {len(ds):,} samples")
    
    combined = ConcatDataset(datasets)
    print(f"  Total: {len(combined):,} samples")
    return combined

print("Building TRAIN dataset:")
train_dataset = build_combined_dataset(train_data, config, PAIRS)

print("\nBuilding VAL dataset:")
val_dataset = build_combined_dataset(val_data, config, PAIRS)

print("\nBuilding TEST dataset:")
test_dataset = build_combined_dataset(test_data, config, PAIRS)

# DataLoaders with GPU-optimized settings
NUM_WORKERS = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
)

print(f"\nDataLoaders ready:")
print(f"  Train: {len(train_loader):,} batches × {config.batch_size}")
print(f"  Val:   {len(val_loader):,} batches")
print(f"  Test:  {len(test_loader):,} batches")
print(f"  Workers: {NUM_WORKERS}, pin_memory={device.type == 'cuda'}")

## 6. Model Architecture

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Instantiate model, print architecture
# ═══════════════════════════════════════════════════════════════
model = WaveFollower(config)
model = model.to(device)

n_params = model.count_parameters()
print(f"WaveFollower Model")
print(f"{'='*50}")
print(f"Total parameters   : {n_params:,}")
print(f"Timeframes         : {config.timeframes}")
print(f"TF wave dim        : {config.tf_wave_dim}")
print(f"Fused dim          : {config.fused_dim}")
print(f"Predictor layers   : {config.predictor_layers}")
print(f"Predictor heads    : {config.predictor_heads}")
print(f"Trend classes      : {config.n_trend_classes} (UP/DOWN/NEUTRAL)")
print(f"Signal classes     : {config.n_signal_classes} (BUY/SELL/HOLD)")
print(f"Device             : {device}")
print()

# Quick forward pass sanity check
sample_batch = next(iter(train_loader))
model_input = {
    tf: {k: v.to(device) for k, v in sample_batch[tf].items()}
    for tf in config.timeframes
    if tf in sample_batch and isinstance(sample_batch[tf], dict)
}
with torch.no_grad():
    out = model(model_input)

print("Output tensors:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:20s}: {list(v.shape)}")
print("\nForward pass OK")

## 7. Training Setup — Optimizer, Loss, Scheduler

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8: Loss, optimizer, scheduler, mixed precision
# ═══════════════════════════════════════════════════════════════
from wavetrader.train_wave_follower import TrendFollowerLoss

criterion = TrendFollowerLoss(
    signal_weight=1.0,   # Main BUY/SELL/HOLD loss
    trend_weight=0.5,    # Trend direction (UP/DOWN/NEUTRAL)
    conf_weight=0.1,     # Confidence calibration
    add_weight=0.2,      # Pullback pyramid scoring
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.epochs,
    eta_min=config.learning_rate * 0.01,
)

# Mixed precision for GPU speedup (2-3x on modern GPUs)
use_amp = device.type == "cuda"
scaler = GradScaler(enabled=use_amp)

print(f"Optimizer     : AdamW (lr={config.learning_rate}, wd=0.01)")
print(f"Scheduler     : CosineAnnealing (T_max={config.epochs})")
print(f"Mixed precision: {'Enabled (float16)' if use_amp else 'Disabled (CPU mode)'}")
print(f"Loss weights  : signal=1.0, trend=0.5, conf=0.1, add=0.2")

## 8. Training Loop — GPU Accelerated with Mixed Precision

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9: Full training loop
# ═══════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, scaler, device, config, use_amp):
    model.train()
    total_loss = 0.0
    correct, n_samples = 0, 0
    loss_components = {"signal": 0, "trend": 0, "conf": 0, "add": 0}
    
    for batch in loader:
        signal_labels = batch["label"].to(device, non_blocking=True)
        trend_labels = batch.get("trend_label")
        if trend_labels is not None:
            trend_labels = trend_labels.to(device, non_blocking=True)
        add_targets = batch.get("add_target")
        if add_targets is not None:
            add_targets = add_targets.to(device, non_blocking=True)
        
        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, signal_labels, trend_labels, add_targets)
        
        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses["total"].item()
        correct += (out["signal_logits"].argmax(-1) == signal_labels).sum().item()
        n_samples += signal_labels.size(0)
        
        loss_components["signal"] += losses["signal_loss"].item()
        loss_components["trend"] += losses["trend_loss"].item()
        loss_components["conf"] += losses["conf_loss"].item()
        loss_components["add"] += losses["add_loss"].item()
    
    n_batches = max(len(loader), 1)
    return {
        "loss": total_loss / n_batches,
        "accuracy": correct / max(n_samples, 1),
        "signal_loss": loss_components["signal"] / n_batches,
        "trend_loss": loss_components["trend"] / n_batches,
        "conf_loss": loss_components["conf"] / n_batches,
        "add_loss": loss_components["add"] / n_batches,
    }


@torch.no_grad()
def validate(model, loader, criterion, device, config, use_amp):
    model.eval()
    total_loss = 0.0
    correct, n_samples = 0, 0
    all_preds, all_labels = [], []
    
    for batch in loader:
        signal_labels = batch["label"].to(device, non_blocking=True)
        trend_labels = batch.get("trend_label")
        if trend_labels is not None:
            trend_labels = trend_labels.to(device, non_blocking=True)
        
        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        
        with autocast(enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, signal_labels, trend_labels)
        
        preds = out["signal_logits"].argmax(-1)
        total_loss += losses["total"].item()
        correct += (preds == signal_labels).sum().item()
        n_samples += signal_labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(signal_labels.cpu().tolist())
    
    n_batches = max(len(loader), 1)
    return {
        "loss": total_loss / n_batches,
        "accuracy": correct / max(n_samples, 1),
        "predictions": all_preds,
        "labels": all_labels,
    }


# ── Main training loop ──────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
best_val_acc = 0.0
best_epoch = 0
checkpoint_path = CHECKPOINT_DIR / "model_weights.pt"

print(f"Training WaveFollower for {config.epochs} epochs")
print(f"{'='*70}")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train Acc':>9} | {'Val Acc':>9} | {'LR':>10} | {'Time':>5}")
print(f"{'-'*70}")

training_start = time.time()

for epoch in range(config.epochs):
    epoch_start = time.time()
    
    # Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, config, use_amp
    )
    
    # Validate
    val_metrics = validate(model, val_loader, criterion, device, config, use_amp)
    
    # Step scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]
    
    # Record history
    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["lr"].append(current_lr)
    
    elapsed = time.time() - epoch_start
    star = ""
    
    # Save best checkpoint
    if val_metrics["accuracy"] > best_val_acc:
        best_val_acc = val_metrics["accuracy"]
        best_epoch = epoch + 1
        star = " *"
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": {
                "timeframes": config.timeframes,
                "lookbacks": config.lookbacks,
                "tf_wave_dim": config.tf_wave_dim,
                "fused_dim": config.fused_dim,
                "predictor_layers": config.predictor_layers,
                "predictor_heads": config.predictor_heads,
                "model_type": "wavefollower",
            },
            "epoch": epoch + 1,
            "val_accuracy": val_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "optimizer_state_dict": optimizer.state_dict(),
        }, str(checkpoint_path))
    
    print(
        f"{epoch+1:5d} | {train_metrics['loss']:10.4f} | {val_metrics['loss']:10.4f} | "
        f"{train_metrics['accuracy']:8.2%} | {val_metrics['accuracy']:8.2%} | "
        f"{current_lr:10.2e} | {elapsed:4.0f}s{star}"
    )

total_time = time.time() - training_start
print(f"{'='*70}")
print(f"Training complete in {total_time/60:.1f} min")
print(f"Best val accuracy: {best_val_acc:.2%} at epoch {best_epoch}")
print(f"Checkpoint: {checkpoint_path}")

## 9. Training Curves & Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10: Plot training curves
# ═══════════════════════════════════════════════════════════════
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Loss", "Accuracy", "Learning Rate", "Train vs Val Gap"),
    vertical_spacing=0.12,
)

epochs_x = list(range(1, len(history["train_loss"]) + 1))

# Loss
fig.add_trace(go.Scatter(x=epochs_x, y=history["train_loss"], name="Train Loss", line=dict(color="#f85149")), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history["val_loss"], name="Val Loss", line=dict(color="#58a6ff")), row=1, col=1)

# Accuracy
fig.add_trace(go.Scatter(x=epochs_x, y=history["train_acc"], name="Train Acc", line=dict(color="#3fb950")), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_x, y=history["val_acc"], name="Val Acc", line=dict(color="#bc8cff")), row=1, col=2)

# Learning Rate
fig.add_trace(go.Scatter(x=epochs_x, y=history["lr"], name="LR", line=dict(color="#d29922")), row=2, col=1)

# Overfit gap (train_acc - val_acc)
gap = [t - v for t, v in zip(history["train_acc"], history["val_acc"])]
fig.add_trace(go.Scatter(x=epochs_x, y=gap, name="Acc Gap", fill="tozeroy", line=dict(color="#f85149")), row=2, col=2)
fig.add_hline(y=0.05, line_dash="dash", line_color="gray", row=2, col=2)

fig.update_layout(
    height=600, width=900,
    title_text="WaveFollower Training Curves",
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#161b22",
    showlegend=False,
)
fig.show()

## 10. Test Set Evaluation & Per-Class Breakdown

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 11: Test evaluation with confusion matrix
# ═══════════════════════════════════════════════════════════════

# Reload best checkpoint
best_state = torch.load(str(checkpoint_path), weights_only=False, map_location=device)
model.load_state_dict(best_state["model_state_dict"])
print(f"Loaded best checkpoint (epoch {best_state['epoch']}, val_acc={best_state['val_accuracy']:.2%})")

# Evaluate on test set
test_metrics = validate(model, test_loader, criterion, device, config, use_amp)

print(f"\nTest Results:")
print(f"  Loss     : {test_metrics['loss']:.4f}")
print(f"  Accuracy : {test_metrics['accuracy']:.2%}")

# Per-class breakdown
from collections import Counter
SIGNAL_NAMES = ["BUY", "SELL", "HOLD"]

preds = test_metrics["predictions"]
labels = test_metrics["labels"]

print(f"\nClass Distribution:")
label_counts = Counter(labels)
pred_counts = Counter(preds)
for i, name in enumerate(SIGNAL_NAMES):
    n_true = label_counts.get(i, 0)
    n_pred = pred_counts.get(i, 0)
    # Per-class accuracy
    class_correct = sum(1 for p, l in zip(preds, labels) if l == i and p == i)
    class_total = label_counts.get(i, 1)
    class_acc = class_correct / max(class_total, 1)
    print(f"  {name:5s}: true={n_true:5d}, pred={n_pred:5d}, acc={class_acc:.2%}")

# Confusion matrix
print(f"\nConfusion Matrix:")
print(f"{'':>10} {'BUY':>8} {'SELL':>8} {'HOLD':>8}")
for i, name in enumerate(SIGNAL_NAMES):
    row = [sum(1 for p, l in zip(preds, labels) if l == i and p == j) for j in range(3)]
    print(f"{name:>10} {row[0]:8d} {row[1]:8d} {row[2]:8d}")

# BUY/SELL-only accuracy (excluding HOLD)
trade_preds = [(p, l) for p, l in zip(preds, labels) if l in (0, 1)]
if trade_preds:
    trade_acc = sum(1 for p, l in trade_preds if p == l) / len(trade_preds)
    print(f"\nBUY/SELL accuracy (excluding HOLD): {trade_acc:.2%} ({len(trade_preds)} samples)")

## 11. Save Final Checkpoint & Summary

Saves the model in the format expected by the dashboard and streaming engine. The checkpoint is saved to `checkpoints/wavefollower/model_weights.pt` — the same path referenced in `docker-compose.yml`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 12: Save final checkpoint + training metadata
# ═══════════════════════════════════════════════════════════════

# Save full checkpoint (model + optimizer + history)
final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "model_type": "wavefollower",
        "timeframes": config.timeframes,
        "lookbacks": config.lookbacks,
        "tf_wave_dim": config.tf_wave_dim,
        "fused_dim": config.fused_dim,
        "predictor_layers": config.predictor_layers,
        "predictor_heads": config.predictor_heads,
        "n_trend_classes": config.n_trend_classes,
        "n_signal_classes": config.n_signal_classes,
        "dropout": config.dropout,
        "pair": "",  # pair-agnostic
    },
    "training": {
        "epochs": config.epochs,
        "learning_rate": config.learning_rate,
        "batch_size": config.batch_size,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_acc,
        "test_accuracy": test_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "pairs_trained_on": list(PAIR_NAMES.values()),
        "total_train_samples": len(train_dataset),
        "total_val_samples": len(val_dataset),
        "total_test_samples": len(test_dataset),
        "training_time_min": total_time / 60,
        "timestamp": datetime.now().isoformat(),
        "device": str(device),
    },
    "history": history,
}

# Save as model_weights.pt (dashboard-compatible name)
torch.save(final_checkpoint, str(checkpoint_path))
print(f"Checkpoint saved: {checkpoint_path}")
print(f"  Size: {checkpoint_path.stat().st_size / 1e6:.1f} MB")

# Also save a copy with timestamp
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
timestamped_path = CHECKPOINT_DIR / f"wavefollower_{ts}.pt"
torch.save(final_checkpoint, str(timestamped_path))
print(f"Timestamped copy: {timestamped_path}")

# Save training history as JSON for later analysis
history_path = CHECKPOINT_DIR / "training_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved: {history_path}")

print(f"\n{'='*50}")
print(f"WaveFollower Training Complete")
print(f"{'='*50}")
print(f"  Parameters     : {model.count_parameters():,}")
print(f"  Best val acc   : {best_val_acc:.2%} (epoch {best_epoch})")
print(f"  Test accuracy  : {test_metrics['accuracy']:.2%}")
print(f"  Training time  : {total_time/60:.1f} min")
print(f"  Pairs trained  : {', '.join(PAIR_NAMES.values())}")
print(f"  OANDA account  : 101-001-30902818-002")
print(f"\nNext steps:")
print(f"  1. Uncomment wavetrader-wavefollower in docker-compose.yml")
print(f"  2. docker compose up -d --build")
print(f"  3. Select 'WaveFollower' in the dashboard dropdown")